In [3]:
"""
Publication figure, built by reusing the SAME panel rendering used in
evaluate_model (the per-model plots that already look right): confusion matrix
with horizontal class names + colourbar, entropy distribution with peak markers,
reliability diagram. Six models are stacked as six rows (4 individual + 2
ensembles) in a plain subplots(6, 3) grid, with tight_layout handling spacing.

Only the subplot TITLES change: a(1), a(2), a(3) ... f(1), f(2), f(3), with the
accuracy added to each confusion-matrix title.

Standalone: loads the predictions already saved to disk, recomputes nothing.
Run once per dataset by changing DATASET. Writes Fig<N>.png (1200 dpi) and
Fig<N>.tif (600 dpi) into ./<dataset>_results/.
"""

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.calibration import calibration_curve

# ======================================================================
# CONFIG  ->  change DATASET only
# ======================================================================
DATASET = 'organamnist'   # 'bloodmnist' | 'breastmnist' | 'dermamnist' | 'organamnist'

DATASET_CONFIG = {
    'bloodmnist':   {'n_classes': 8,  'figure_number': 3},
    'breastmnist':  {'n_classes': 2,  'figure_number': 5},
    'dermamnist':   {'n_classes': 7,  'figure_number': 7},
    'organamnist':  {'n_classes': 11, 'figure_number': 9},
}

CLASS_NAMES_MAP_4CM = {
    'bloodmnist': ['basophil', 'eosinophil', 'erythroblast', 'immature \ngranulocyte',
                   'lymphocyte', 'monocyte', 'neutrophil', 'platelet'],
    'breastmnist': ['malignant', 'normal/benign'],
    'dermamnist': ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc'],
    'organamnist': ['bladder', 'femur-L', 'femur-R', 'heart', 'kidney-L', 'kidney-R',
                    'liver', 'lung-L', 'lung-R', 'pancreas', 'spleen'],
}

n_classes       = DATASET_CONFIG[DATASET]['n_classes']
FIGURE_NUMBER   = DATASET_CONFIG[DATASET]['figure_number']
CLASS_NAMES_4CM = CLASS_NAMES_MAP_4CM[DATASET]

INPUT_DIR  = f"./{DATASET}_outputs"
OUTPUT_DIR = f"./{DATASET}_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

INDIVIDUAL_MODELS = ['ConvNeXtBase', 'ViT-Base', 'EfficientNetV2M', 'InceptionResNetV2']

# Confusion-matrix colours: same colourmap for all individual models,
# distinct ones for the two ensembles.
CM_CMAPS = {
    'ConvNeXtBase':      'Oranges',
    'ViT-Base':          'Oranges',
    'EfficientNetV2M':   'Oranges',
    'InceptionResNetV2': 'Oranges',
    'SoftVoting':        'Blues',
    'RigorousStacking':  'Greens',
}

# Row order and letters
ROW_MODELS  = INDIVIDUAL_MODELS + ['SoftVoting', 'RigorousStacking']
ROW_LETTERS = list("abcdef")

# Optional, for Springer: uncomment to force Arial (your laptop has it).
# plt.rcParams['font.family'] = 'Arial'

# ======================================================================
# LOAD STORED PREDICTIONS (no recomputation)
# ======================================================================
def _require(path, what):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Could not find {what} at {path}. "
                                f"Check that DATASET='{DATASET}' matches your folders.")
    return path

# ground truth (test)
for _f in ('y_true_flat.npy', 'y_test_true.npy'):
    _p = os.path.join(INPUT_DIR, _f)
    if os.path.exists(_p):
        y_true_flat = np.load(_p).flatten()
        break
else:
    raise FileNotFoundError(f"Could not find test ground truth in {INPUT_DIR}")

# individual models (with the same NaN repair the notebook used)
test_preds = {}
for m in INDIVIDUAL_MODELS:
    arr = np.load(_require(os.path.join(INPUT_DIR, f"{m}_test_preds.npy"), f"{m} test preds"))
    mask = np.any(np.isnan(arr), axis=1)
    if mask.sum() > 0:
        arr[mask] = 1.0 / n_classes
    test_preds[m] = arr

# ensembles
all_probs = dict(test_preds)
all_probs['SoftVoting']      = np.load(_require(os.path.join(OUTPUT_DIR, "SoftVoting_test_preds.npy"), "SoftVoting preds"))
all_probs['RigorousStacking'] = np.load(_require(os.path.join(OUTPUT_DIR, "RigorousStacking_test_preds.npy"), "RigorousStacking preds"))

print(f"Loaded {DATASET}: {len(y_true_flat)} test samples, {n_classes} classes.")

# ======================================================================
# ENTROPY PEAKS (top mode of each curve; same logic as analyze_entropy_peaks)
# ======================================================================
def top_entropy_peaks(entropy_correct, entropy_wrong, n_points=1000):
    if len(entropy_correct) < 2 or len(entropy_wrong) < 1:
        return None
    x_grid = np.linspace(0, max(entropy_correct.max(), entropy_wrong.max()) + 0.1, n_points)
    dc = gaussian_kde(entropy_correct)(x_grid)
    ic = int(np.argmax(dc))
    pcx, pcy = float(x_grid[ic]), float(dc[ic])
    if len(entropy_wrong) >= 2:
        dw = gaussian_kde(entropy_wrong)(x_grid)
        iw = int(np.argmax(dw))
        pwx, pwy = float(x_grid[iw]), float(dw[iw])
    else:
        pwx, pwy = float(entropy_wrong[0]), 0.0
    return pcx, pcy, pwx, pwy

# ======================================================================
# DRAW ONE ROW = the three evaluate_model panels, titled letter(1/2/3)
# ======================================================================
def draw_row(axes_row, model_name, letter):
    probs = all_probs[model_name]
    y_pred = np.argmax(probs, axis=1)
    acc = accuracy_score(y_true_flat, y_pred)
    ax_cm, ax_ent, ax_rel = axes_row

    # (1) Confusion matrix  (class names, colourbar, per-model colourmap)
    cm = confusion_matrix(y_true_flat, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=CM_CMAPS[model_name], ax=ax_cm,
                xticklabels=CLASS_NAMES_4CM, yticklabels=CLASS_NAMES_4CM)
    ax_cm.set_title(f'{letter}(1)   Acc = {acc:.2%}')
    ax_cm.set_xlabel('Predicted')
    ax_cm.set_ylabel('True')
    ax_cm.tick_params(axis='x', labelsize=7)
    ax_cm.tick_params(axis='y', labelsize=7)
    plt.setp(ax_cm.get_xticklabels(), rotation=0, ha='center', rotation_mode='anchor')

    # (2) Entropy distribution with peak markers
    entropy = -np.sum(probs * np.log(probs + 1e-9), axis=1)
    correct_mask = (y_pred == y_true_flat)
    sns.kdeplot(entropy[correct_mask], label=f'Correct (n={correct_mask.sum()})',
                fill=True, color='#2166AC', ax=ax_ent)
    sns.kdeplot(entropy[~correct_mask], label=f'Wrong (n={(~correct_mask).sum()})',
                fill=True, color='#D97B00', ax=ax_ent)
    peaks = top_entropy_peaks(entropy[correct_mask], entropy[~correct_mask])
    if peaks:
        pcx, pcy, pwx, pwy = peaks
        ax_ent.axvline(pcx, color='#2166AC', linestyle=':', alpha=0.6)
        ax_ent.annotate(f'{pcx:.2f}\nh={pcy:.2f}', xy=(pcx, pcy), xytext=(5, 5),
                        textcoords='offset points', fontsize=8, color='#2166AC')
        ax_ent.axvline(pwx, color='#D97B00', linestyle=':', alpha=0.6)
        ax_ent.annotate(f'{pwx:.2f}\nh={pwy:.2f}', xy=(pwx, pwy), xytext=(5, -15),
                        textcoords='offset points', fontsize=8, color='#D97B00')
    ax_ent.set_title(f'{letter}(2)')
    ax_ent.set_xlabel('Shannon Entropy')
    ax_ent.legend()

    # (3) Reliability diagram
    is_correct = correct_mask.astype(int)
    conf = np.max(probs, axis=1)
    prob_true, prob_pred = calibration_curve(is_correct, conf, n_bins=10, strategy='quantile')
    ax_rel.plot(prob_pred, prob_true, marker='o', label=model_name, color='purple')
    ax_rel.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect')
    ax_rel.set_title(f'{letter}(3)')
    ax_rel.set_xlabel('Confidence')
    ax_rel.set_ylabel('Accuracy')
    ax_rel.legend()
    ax_rel.grid(True, alpha=0.3)

# ======================================================================
# BUILD AND SAVE
# ======================================================================
plt.rcParams['svg.fonttype'] = 'path'   # text becomes vector outlines: identical on any viewer

def generate_figure(figsize=(20, 30), png_dpi=1200, png_dpi_900=900, tiff_dpi=600):
    fig, axes = plt.subplots(6, 3, figsize=figsize)
    fig.patch.set_facecolor('white')          # opaque white canvas, no transparency
    for i, model_name in enumerate(ROW_MODELS):
        draw_row(axes[i], model_name, ROW_LETTERS[i])
    plt.tight_layout()

    png_path     = f"{OUTPUT_DIR}/Fig{FIGURE_NUMBER}.png"
    png_path_900 = f"{OUTPUT_DIR}/Fig{FIGURE_NUMBER}_900.png"
    tif_path     = f"{OUTPUT_DIR}/Fig{FIGURE_NUMBER}.tif"
    svg_path     = f"{OUTPUT_DIR}/Fig{FIGURE_NUMBER}.svg"

    fig.savefig(png_path,     dpi=png_dpi,     bbox_inches='tight',
                facecolor='white', edgecolor='none')
    fig.savefig(png_path_900, dpi=png_dpi_900, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    fig.savefig(tif_path,     dpi=tiff_dpi,    bbox_inches='tight',
                facecolor='white', edgecolor='none')

    # Vector SVG: highest quality, resolution-independent, opaque white background.
    fig.savefig(svg_path, format='svg', bbox_inches='tight',
                facecolor='white', edgecolor='none', transparent=False)

    plt.close(fig)
    print(f"Saved {png_path}  ({png_dpi} dpi)")
    print(f"Saved {png_path_900}  ({png_dpi_900} dpi)")
    print(f"Saved {tif_path}  ({tiff_dpi} dpi)")
    print(f"Saved {svg_path}  (vector)")

if __name__ == "__main__":
    generate_figure()

Loaded organamnist: 17778 test samples, 11 classes.
Saved ./organamnist_results/Fig9.png  (1200 dpi)
Saved ./organamnist_results/Fig9_900.png  (900 dpi)
Saved ./organamnist_results/Fig9.tif  (600 dpi)
Saved ./organamnist_results/Fig9.svg  (vector)
